In [ ]:
#!/usr/bin/env python3
import re
from pathlib import Path

# --- Configuration ---
PROJECT_DIR = Path("..").resolve()

# --- Step 1: Collect all .tscn files ---
# all_scenes = {p.relative_to(PROJECT_DIR).as_posix() for p in PROJECT_DIR.rglob("*.tscn")}
# exclude = set(PROJECT_DIR / p for p in ["DevTools", ".github", ".godot"])
exclude = set(["DevTools", ".github", ".godot"])
# all_scenes = {p.relative_to(PROJECT_DIR).as_posix() for p in PROJECT_DIR.rglob("*.png") if not any(p.is_relative_to(e) for e in exclude)}
all_scenes = {p.relative_to(PROJECT_DIR).as_posix() for p in PROJECT_DIR.rglob("*.png") if p.relative_to(PROJECT_DIR).parts[0] not in exclude}

# --- Step 2: Search for .tscn references in text files (.gd, .tscn, .cfg, etc.) ---
used_scenes = set()
# tscn_pattern = re.compile(r'["\']res://(.*?\.tscn)["\']')
tscn_pattern = re.compile(r'["\']res://(.*?\.png)["\']')

for path in PROJECT_DIR.rglob("*"):
  if path.relative_to(PROJECT_DIR).parts[0] in set(["DevTools", ".github", ".godot"]):
    continue
  if path.suffix in (".gd", ".tscn", ".cfg", ".tres"): # ".import"
    try:
      text = path.read_text(encoding="utf-8")
    except Exception:
      continue  # skip unreadable files (e.g., binary .import)
    for match in tscn_pattern.findall(text):
      used_scenes.add(match)

# --- Step 3: Compare sets to find unused scenes ---
unused = sorted(all_scenes - used_scenes)

# --- Step 4: Print results ---
print(f"\nFound {len(unused)} potentially unused .tscn files:\n")
for scene in unused:
  print(f"  - {scene}")

if not unused:
  print("✅ No abandoned scenes found!")
else:
  print("\n⚠️  Review these before deleting — some might be used dynamically.")
